In [1]:
using Healpix
using LinearAlgebra
using StaticArrays
using Base.Threads
using WignerD
using NPZ
using Plots
#using PyPlot
#using PyCall
using BenchmarkTools
#hp = pyimport("healpy")
#include("../src/EBC.jl")

In [2]:
mutable struct ConvolutionSky{I<:Int, MC<:Matrix{Complex{Float64}}}
    nside::I
    lmax::I
    alm::MC
    realization::I
end

mutable struct ConvolutionBeam{I<:Int, MC<:Matrix{Complex{Float64}}, Bl<:Bool}
    nside::I
    lmax::I
    beammax::I
    blm::MC
    HWP::Bl
end

In [6]:
function ConvolutionSky(;
        nside::Int = 128,
        lmax::Int = 3*nside - 1,
        alm::Matrix{ComplexF64} = fill(1.0 + 1.0im, 2, 2),
        realization::Int = 1
    )
    return ConvolutionSky(
        nside,
        lmax,
        alm,
        realization
    )
end

function ConvolutionBeam(;
        nside::Int =12*128^2,
        lmax::Int =3*nside-1,
        beammax::Int = 2,
        blm::Matrix{ComplexF64} = [1.0+1im 1.0+1im;1.0+1im 1.0+1im],
        HWP::Bool = false
    )
    return ConvolutionBeam(
        nside,
        lmax,
        beammax,
        blm,
        HWP
    )
end

ConvolutionBeam

In [7]:
cs = ConvolutionSky()
cb = ConvolutionBeam()

ConvolutionBeam{Int64, Matrix{ComplexF64}, Bool}(196608, 589823, 2, ComplexF64[1.0 + 1.0im 1.0 + 1.0im; 1.0 + 1.0im 1.0 + 1.0im], false)

In [8]:
cb.HWP

false

In [4]:
function ConvolutionSky(;
    nside::Int = 128,
    lmax::Int = 3*nside - 1,
    alm::Matrix{ComplexF64} = fill(1.0 + 1.0im, 2, 2),
    realization::Int = 1
)
    return ConvolutionSky(nside, lmax, alm, realization)
end

ConvolutionSky

In [25]:
mutable struct ConvolutionSky{I<:Int, MC<:Matrix{Complex{Float64}}}
    nside::I
    lmax::I
    alm::MC
    realization::I
end

LoadError: invalid redefinition of constant Main.ConvolutionSky

In [5]:
nside = 16
cp = gen_ConvolutionParams_pc(nside = nside);
cp.beam_mmax = 10

10

In [6]:
lmax = nside*3-1
res = Resolution(nside)
beammmax = 10

10

In [7]:
# input beam
size = alm_idx(lmax,lmax,lmax)
blm_ = zeros(ComplexF64, 3, size)
blm = hp.blm_gauss(deg2rad(0.5), lmax = lmax, pol = true)
blm_[1,1:length(blm[1,:])] = blm[1,:]
blm_[2,1:length(blm[1,:])] = -blm[2,:]*sqrt(2)
blm_[3,1:length(blm[1,:])] = -blm[3,:]*sqrt(2);
@time blm_full = get_reorder_blm_optimized(blm_, lmax, beammmax);

LoadError: UndefVarError: `hp` not defined

In [8]:
alm = npzread("./inputs/alm=CMB=r0=nside$nside.npy")
@time alm_full = get_reorder_alm_optimized(alm, lmax);

  0.233391 seconds (245.43 k allocations: 16.078 MiB, 3.35% gc time, 98.94% compilation time)


In [9]:
data_path = "/data/n/n339/wigner_file/"

"/data/n/n339/wigner_file/"

In [10]:
@time initial_wignerd = [zeros(ComplexF64,2*i+1,2*i+1) for i in 0:cp.lmax]
@time initial_wignerd_ = [zeros(ComplexF64,2*i+1,2*i+1) for i in 0:cp.lmax]
@time eff_wignerD = [zeros(ComplexF64,2*i+1,2*i+1) for i in 0:cp.lmax];
@time eff_wignerD_ = [zeros(ComplexF64,2*i+1,2*i+1) for i in 0:cp.lmax];

  0.071945 seconds (29.82 k allocations: 4.259 MiB, 40.88% gc time, 57.19% compilation time)
  0.029705 seconds (21.82 k allocations: 3.724 MiB, 97.19% compilation time)
  0.028619 seconds (21.82 k allocations: 3.724 MiB, 96.99% compilation time)
  0.036557 seconds (21.82 k allocations: 3.724 MiB, 96.14% compilation time)


In [11]:
initial_wignerd[2]

3×3 Matrix{ComplexF64}:
 0.0+0.0im  0.0+0.0im  0.0+0.0im
 0.0+0.0im  0.0+0.0im  0.0+0.0im
 0.0+0.0im  0.0+0.0im  0.0+0.0im

In [12]:
function initialwignerds_array_(cp, theta, path, initial_wignerd)
    count = 1
    for l in cp.l_range[1]:cp.l_range[2]
        initial_wignerd[count] = swignerd_calc(l, theta, path)
        count += 1
    end
   return initial_wignerd
end

initialwignerds_array_ (generic function with 1 method)

In [13]:
function effective_wignerD_onz_(cp, ψs, initial_wignerd_onz_)
    for m in -cp.lmax-2:cp.lmax+2
        initial_wignerd_onz_[cp.lmax+1+m] = mean(exp.(-1im*ψs*m))
    end
    return initial_wignerd_onz_
end

effective_wignerD_onz_ (generic function with 1 method)

In [14]:
npix = nside2npix(nside)
utv = unique_theta_val(cp);

In [15]:
initial_wignerd_onz = zeros(ComplexF64, 3, 2*cp.lmax+1)
conv_binned_map = zeros(3,npix)
result_d = zeros(ComplexF64,3)
@time for i in 1:length(utv)
    @show i
    start_pix, stop_pix = unique_theta_num(i, cp)
    initialwignerds_array(cp, utv[i], data_path, initial_wignerd);
    for pix_num in start_pix:stop_pix
        center_th, center_phi = pix2angRing(res, pix_num)
        calc_psi = rand(100)*2pi
        effective_wignerD_onz(cp, calc_psi[:], initial_wignerd_onz)
        for j in 1:3
            get_pc_total_effective_wignerD(cp, center_phi, initial_wignerd_onz[j,:], initial_wignerd, eff_wignerD);
            result_d[j] = eff_convolver_optimized(alm_full, blm_full, eff_wignerD, beammmax)
        end
        conv = binned_mapmaker(calc_psi, result_d)
        conv_binned_map[:,pix_num] .= conv
    end
end

i = 1


LoadError: SystemError: opening file "/data/n/n339/wigner_file/dmatrices=ell0.npy": No such file or directory

In [ ]:
alm_smooth = hp.smoothalm([alm[1,:],alm[2,:],alm[3,:]],fwhm = deg2rad(0.5))
in_map = hp.alm2map([alm_smooth[1],alm_smooth[2],alm_smooth[3]], nside = nside)

LoadError: UndefVarError: `alm` not defined